In [ ]:
# Problema: Actualizar una tabla analítica con un lote incremental sin duplicar registros ya procesados.

"""Aplica un lote incremental con checkpoint e idempotencia visible."""
import json
from pathlib import Path
import pandas as pd
ROOT=next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir()); OUTPUT=ROOT/"submission/customers_current.parquet"; REPORT=ROOT/"submission/incremental_report.csv"; CHECKPOINT=ROOT/"temp/checkpoint.json"
def build_submission():
    current=pd.DataFrame([["C1","Ana","Basic","2026-09-01T09:00:00"],["C2","Luis","Premium","2026-09-01T09:00:00"],["C3","Mariana","Basic","2026-09-01T09:00:00"]],columns=["customer_id","customer_name","segment","updated_at"])
    changes=pd.DataFrame([["C2","Luis Gomez","Corporate","2026-09-16T10:00:00"],["C3","Mariana","Basic","2026-09-01T09:00:00"],["C4","Diego","Premium","2026-09-16T10:30:00"]],columns=current.columns)
    actions=[]
    for row in changes.itertuples(index=False):
        exists=current.customer_id.eq(row.customer_id)
        if not exists.any(): actions.append("INSERT"); current=pd.concat([current,pd.DataFrame([row],columns=current.columns)],ignore_index=True)
        elif current.loc[exists,"updated_at"].iloc[0] < row.updated_at: actions.append("UPDATE"); current.loc[exists]=list(row)
        else: actions.append("UNCHANGED")
    current.sort_values("customer_id").to_parquet(OUTPUT,index=False)
    pd.DataFrame([[a,actions.count(a)] for a in ["INSERT","UPDATE","UNCHANGED"]],columns=["action","record_count"]).to_csv(REPORT,index=False)
    CHECKPOINT.write_text(json.dumps({"high_water_mark":"2026-09-16T10:30:00"}))
if __name__=="__main__":build_submission()